# Validação 07 — Recuperação lexical de evidências

## Goal

Comprovar que uma linha de base BM25 recupera e ordena trechos científicos relacionados à alegação, preservando a proveniência de cada resultado.

## Setup

O notebook recupera o artigo `PMC7805365` pelas fontes oficiais, cria os trechos rastreáveis da etapa anterior e consulta o índice lexical implementado no módulo real da aplicação.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    Bm25Index,
    ChunkingConfig,
    PmcClient,
    PubMedClient,
    chunk_article_content,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-23T14:00:24.941945+00:00


## Steps

Primeiro executamos o fluxo já validado para obter o texto completo e seus trechos. Em seguida, o BM25 pontua os trechos pelos termos da consulta.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["33431520[pmid]"]

analysis_input = validate_analysis_input(
    "O consumo de café altera o risco de câncer de próstata."
)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(
        email=os.getenv("NCBI_EMAIL"),
        api_key=os.getenv("NCBI_API_KEY"),
    ),
    max_results_per_query=1,
)
content = retrieve_article_content(
    pubmed_result.publications[0],
    PmcClient(
        email=os.getenv("NCBI_EMAIL"),
        api_key=os.getenv("NCBI_API_KEY"),
    ),
)
chunks = chunk_article_content(
    content,
    ChunkingConfig(max_words=120, overlap_words=20),
)
print(f"Índice preparado com {len(chunks)} trechos de {content.pmcid}.")

Índice preparado com 35 trechos de PMC7805365.


In [3]:
retrieval_query = "coffee consumption prostate cancer risk"
index = Bm25Index(chunks)
results = index.search(retrieval_query, top_k=5)

print(f"Consulta: {retrieval_query}")
print(f"Resultados exibidos: {len(results)}")
for result in results:
    print("\n" + "─" * 80)
    pprint({
        "rank": result.rank,
        "score": round(result.score, 4),
        "section": result.chunk.section,
        "matched_terms": result.matched_terms,
        "chunk_id": result.chunk.chunk_id,
        "text_preview": result.chunk.text[:400],
        "source_url": result.chunk.source_url,
    })

Consulta: coffee consumption prostate cancer risk
Resultados exibidos: 5

────────────────────────────────────────────────────────────────────────────────
{'chunk_id': '33431520:1:3:780ab10c7a21b35b',
 'matched_terms': ('coffee', 'consumption', 'prostate', 'cancer', 'risk'),
 'rank': 1,
 'score': 2.2591,
 'section': 'Introduction',
 'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/',
 'text_preview': 'with the risk of prostate cancer. Although earlier cohort '
                 'studies did not detect an association,7–15 more recent '
                 'studies conducted in major Western countries such as the '
                 'USA, Sweden and the UK reported that coffee consumption was '
                 'associated with a lower risk of localised and advanced '
                 'prostate cancer.16–20 In Japan, a country with increasing '
                 'popularity of coffee, a cohort study also found a signifi'}

────────────────────────────────────────────────────────

## Checks

As verificações confirmam ordenação, resultados únicos, correspondências positivas, repetibilidade e proveniência.

In [4]:
assert 1 <= len(results) <= 5
assert [result.rank for result in results] == list(range(1, len(results) + 1))
assert all(
    previous.score >= current.score
    for previous, current in zip(results, results[1:])
)
assert len({result.chunk.chunk_id for result in results}) == len(results)
assert all(result.score > 0 for result in results)
assert all(result.matched_terms for result in results)
assert all(result.chunk.pmid == "33431520" for result in results)
assert all(result.chunk.pmcid == "PMC7805365" for result in results)
assert all(result.chunk.source_url == content.pmc_url for result in results)

matched_terms = {term for result in results for term in result.matched_terms}
assert {"coffee", "prostate", "cancer"} <= matched_terms

second_run = index.search(retrieval_query, top_k=5)
assert [result.chunk.chunk_id for result in results] == [
    result.chunk.chunk_id for result in second_run
]

print(
    f"Validação aprovada: {len(results)} trechos ordenados; "
    f"primeiro resultado na seção '{results[0].chunk.section}'."
)

Validação aprovada: 5 trechos ordenados; primeiro resultado na seção 'Introduction'.


## Next Steps

A recuperação lexical estará validada quando todas as células forem executadas sem erros. A próxima etapa será adicionar recuperação semântica e comparar sua contribuição antes de combiná-la com o BM25.